# Libraries

In [2]:
import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
import requests
import time
from datetime import datetime, timedelta

# Get Billboard Top 100

In [4]:
def scrape_billboard(date_str):
    url = f"https://www.billboard.com/charts/hot-100/{date_str}"
    response = requests.get(url)
    if response.status_code != 200:
        print(f"❌ Failed to load Billboard for {date_str}")
        return []

    soup = BeautifulSoup(response.text, 'html.parser')
    songs = []

    # Each song entry is inside this list item
    entries = soup.select("li.o-chart-results-list__item")

    for entry in entries:
        title_elem = entry.select_one("h3.c-title")
        artist_elem = entry.select_one("span.c-label")

        if title_elem and artist_elem:
            title = title_elem.get_text(strip=True)
            artist = artist_elem.get_text(strip=True)

            # Filter out ranks, NEW, RE-ENTRY, etc.
            if title and artist and not artist[0].isdigit() and artist.upper() != "NEW":
                songs.append((title, artist))

    return songs

# --- Set your start and end dates here ---
start_date = datetime(1980, 1, 1)
end_date = datetime(1990, 1, 1)

# --- Generate all weekly dates between start and end ---
dates = []
current = start_date
while current <= end_date:
    dates.append(current.strftime("%Y-%m-%d"))
    current += timedelta(weeks=1)

# --- Scrape Billboard for each week ---
billboard_data = {}  # Billboard songs per date
unique_songs = set()  # Deduplicated (title, artist) pairs

for date in dates:
    print(f"📅 Scraping Billboard for {date}...")
    chart = scrape_billboard(date)
    billboard_data[date] = chart
    for title, artist in chart:
        unique_songs.add((title, artist))  # Deduplicate

# --- Final Output ---
print(f"\n🎧 Total unique songs across {len(dates)} weeks in the 80s: {len(unique_songs)}")

📅 Scraping Billboard for 1980-01-01...
📅 Scraping Billboard for 1980-01-08...
📅 Scraping Billboard for 1980-01-15...
📅 Scraping Billboard for 1980-01-22...
📅 Scraping Billboard for 1980-01-29...
📅 Scraping Billboard for 1980-02-05...
📅 Scraping Billboard for 1980-02-12...
📅 Scraping Billboard for 1980-02-19...
📅 Scraping Billboard for 1980-02-26...
📅 Scraping Billboard for 1980-03-04...
📅 Scraping Billboard for 1980-03-11...
📅 Scraping Billboard for 1980-03-18...
📅 Scraping Billboard for 1980-03-25...
📅 Scraping Billboard for 1980-04-01...
📅 Scraping Billboard for 1980-04-08...
📅 Scraping Billboard for 1980-04-15...
📅 Scraping Billboard for 1980-04-22...
📅 Scraping Billboard for 1980-04-29...
📅 Scraping Billboard for 1980-05-06...
📅 Scraping Billboard for 1980-05-13...
📅 Scraping Billboard for 1980-05-20...
📅 Scraping Billboard for 1980-05-27...
📅 Scraping Billboard for 1980-06-03...
📅 Scraping Billboard for 1980-06-10...
📅 Scraping Billboard for 1980-06-17...
📅 Scraping Billboard for 

# Get MBIDs of each unique song using the MusicBrainz API

In [6]:
import re

def clean_artist_name_variants(artist):
    variants = set()

    # Original version
    variants.add(artist.strip())

    # Common substitutions
    clean = re.sub(r"\b(featuring|feat\.|with|ft\.|and)\b", "&", artist, flags=re.IGNORECASE)
    clean = re.sub(r"\s*&\s*", " & ", clean).strip()
    clean = re.sub(r"\s{2,}", " ", clean)
    variants.add(clean)

    # Remove collaborators (just main artist)
    main_only = re.split(r"\b(featuring|feat\.|with|ft\.|and|&)\b", artist, flags=re.IGNORECASE)[0].strip()
    variants.add(main_only)

    return list(variants)

In [7]:
def get_mbid(title, artist):
    name_variants = clean_artist_name_variants(artist)

    url = "https://musicbrainz.org/ws/2/recording/"
    headers = {
        'User-Agent': 'BillboardMBIDFetcher/1.0 (your-email@example.com)'
    }

    for name in name_variants:
        query = f'"{title}" AND artist:"{name}"'
        params = {
            'query': query,
            'fmt': 'json',
            'limit': 1
        }
        try:
            response = requests.get(url, params=params, headers=headers)
            data = response.json()
            recordings = data.get("recordings", [])
            if recordings:
                # print(f"✅ Found MBID for '{title}' using artist variant: '{name}'")
                return recordings[0]["id"]
        except Exception as e:
            print(f"❌ Error while searching with artist '{name}': {e}")

    return None

## Loop through unique song list:

In [9]:
billboard_with_mbid = []

for title, artist in unique_songs:
    print(f"🔍 {title} — {artist}")
    mbid = get_mbid(title, artist)
    if mbid:
        billboard_with_mbid.append({
            "title": title,
            "artist": artist,
            "mbid": mbid
        })
    else:
        print(f"⚠️ No MBID found for: {title} — {artist}")
    time.sleep(1)

print(f"\n🎧 Total unique song MBIDs across all weeks: {len(billboard_with_mbid)}")

🔍 I Wouldn't Have Missed It For The World — Ronnie Milsap
🔍 Do You Want Crying — Katrina And The Waves
🔍 Radar Love — White Lion
🔍 Me So Horny — The 2 Live Crew
🔍 Long And Lasting Love (Once In A Lifetime) — Glenn Medeiros
🔍 Midnight Rain — Poco
🔍 Life Gets Better — Graham Parker
🔍 Ever Since The World Began — Tommy Shaw
🔍 Everybody's Got To Learn Sometime — The Korgis
🔍 Fire Woman — The Cult
🔍 I Don't Know — Michael Morales
🔍 The Sun Ain't Gonna Shine Anymore — Nielsen/Pearson
⚠️ No MBID found for: The Sun Ain't Gonna Shine Anymore — Nielsen/Pearson
🔍 Honey, Honey — David Hudson
🔍 I Love My Truck — Glen Campbell
🔍 Here With Me — REO Speedwagon
🔍 So Bad — Paul McCartney
🔍 Take No Prisoners (In The Game Of Love) — Peabo Bryson
🔍 Rock Wit'cha — Bobby Brown
🔍 Outstanding — The Gap Band
🔍 Just So Lonely — Get Wet
🔍 Magical — John Parr
🔍 Girl With The Hungry Eyes — Jefferson Starship
🔍 Go For It — Kim Wilde
🔍 Over You — Roxy Music
🔍 Juicy Fruit — Mtume
🔍 Wouldn't It Be Good — Nik Kershaw
🔍 

#  Code to get all the features from AcousticBrainz API

## "AcousticBrainz was a joint effort between **Music Technology Group at Universitat Pompeu Fabra** in Barcelona and the MusicBrainz project. At the heart of this project lies the **Essentia toolkit** from the MTG – this open source toolkit enables the automatic analysis of music. The output from Essentia is collected by the AcousticBrainz project and made available to the public."

### https://essentia.upf.edu/

### https://acousticbrainz.org/data

### "The low-level data includes acoustic descriptors characterizing overall loudness, dynamics and spectral shape of a signal, rhythm descriptors (including beats per minute), and tonal information (including keys and scales). The low-level data is stable, compared to the higher-level data. This hopefully means that we will not have to recompute this data from audio files too frequently."

### "The high-level data includes information about moods, genres, vocals and music type automatically inferred from low-level data by pre-trained classifiers. The high-level data can be computed from the low-level data, without requiring access to original audio files. The high-level data is less stable than the low-level data, since it contains more experimental algorithms."

In [16]:
def get_all_acousticbrainz_features(mbid):
    base_url = "https://acousticbrainz.org/api/v1"
    features = {}

    try:
        # High-level
        hl_url = f"{base_url}/{mbid}/high-level"
        hl_response = requests.get(hl_url)
        if hl_response.status_code == 200:
            features["highlevel"] = hl_response.json()

        # Low-level
        ll_url = f"{base_url}/{mbid}/low-level"
        ll_response = requests.get(ll_url)
        if ll_response.status_code == 200:
            features["lowlevel"] = ll_response.json()

    except Exception as e:
        print(f"❌ Error fetching features for {mbid}: {e}")

    return features if features else None

## Loop through MBIDs:

In [20]:
billboard_with_all_features = []

for track in billboard_with_mbid:
    mbid = track["mbid"]
    print(f"🎧 Fetching full features for: {track['title']} — {track['artist']}")
    features = get_all_acousticbrainz_features(mbid)
    if features:
        track["features"] = features
        billboard_with_all_features.append(track)
    else:
        print(f"⚠️ No features found for MBID: {mbid}")
    time.sleep(1)  # Respect API limits
print(f"\n🎧 Total unique song features across all weeks during the 80s: {len(billboard_with_all_features)}")

🎧 Fetching full features for: I Wouldn't Have Missed It For The World — Ronnie Milsap
⚠️ No features found for MBID: 5fed76d3-f073-4206-819c-56307bce11d5
🎧 Fetching full features for: Do You Want Crying — Katrina And The Waves
🎧 Fetching full features for: Radar Love — White Lion
⚠️ No features found for MBID: 0a3354d9-c04d-4a07-8317-eb3a95068839
🎧 Fetching full features for: Me So Horny — The 2 Live Crew
⚠️ No features found for MBID: 684aa4c1-7347-4e9b-90e0-42cebd58cb01
🎧 Fetching full features for: Long And Lasting Love (Once In A Lifetime) — Glenn Medeiros
⚠️ No features found for MBID: f6c5fa2a-cc2c-468a-9428-e0c2eddbd283
🎧 Fetching full features for: Midnight Rain — Poco
⚠️ No features found for MBID: 3a01303d-fe11-497f-bc21-f88e6327d1df
🎧 Fetching full features for: Life Gets Better — Graham Parker
🎧 Fetching full features for: Ever Since The World Began — Tommy Shaw
🎧 Fetching full features for: Everybody's Got To Learn Sometime — The Korgis
🎧 Fetching full features for: Fire W

# Save as a json file

In [22]:
import json

with open("billboard_80s_all_features.json", "w") as f:
    json.dump(billboard_with_all_features, f, indent=2)

In [29]:
def flatten_dict(d, parent_key='', sep='.'):
    # Recursively flattens a nested dictionary
    items = []
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            items.extend(flatten_dict(v, new_key, sep=sep).items())
        else:
            items.append((new_key, v))
    return dict(items)

def flatten_track_features(track):
    flat = {
        "title": track["title"],
        "artist": track["artist"],
        "mbid": track["mbid"]
    }

    features = track.get("features", {})

    # Flatten high-level and low-level dictionaries
    highlevel_flat = flatten_dict(features.get("highlevel", {}), parent_key="hl")
    lowlevel_flat = flatten_dict(features.get("lowlevel", {}), parent_key="ll")

    # Merge all into one flat dict
    flat.update(highlevel_flat)
    flat.update(lowlevel_flat)
    return flat

# Flatten all songs
rows = [flatten_track_features(track) for track in billboard_with_all_features]

# Create DataFrame and export to CSV
df = pd.DataFrame(rows)
df.to_csv("billboard_80s_features_full.csv", index=False)

print("✅ Full feature set exported to 'billboard_80s_features_full.csv'")


✅ Full feature set exported to 'billboard_80s_features_full.csv'
